# Legal Document Parsing

This notebook transforms cleaned page text into structured legal articles. It detects the CRMP hierarchy, tracks the active part, title, and chapter, and associates each article with its text and source-page range.

The output is `data/processed/crmp_articles.json`, a structured article collection ready for chunking.


## 1. Import the required libraries

Load file, JSON, and regular-expression utilities for rule-based document parsing.


In [1]:
from pathlib import Path
import json
import re


## 2. Define the input and output paths

Locate the cleaned page dataset and set the destination for structured articles.


In [2]:
# Run this notebook from notebooks/ so the project root is its parent.
PROJECT_ROOT = Path.cwd().parent

INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_cleaned.json"
)

OUTPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "crmp_articles.json"
)

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")


Input:  c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_cleaned.json
Output: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_articles.json


## 3. Load the cleaned pages

Read the normalized page records generated by the preprocessing stage.


In [3]:
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

pages = data["pages"]

print(f"Páginas carregadas: {len(pages)}")


Páginas carregadas: 662


## 4. Define legal heading patterns

Compile expressions for parts, titles, chapters, articles, and article subtitles.


In [4]:
PART_PATTERN = re.compile(
    r"^PARTE\s+([A-Z])$",
    re.IGNORECASE
)

TITLE_PATTERN = re.compile(
    r"^T[IÍ]TULO\s+([IVXLCDM]+)$",
    re.IGNORECASE
)

CHAPTER_PATTERN = re.compile(
    r"^CAP[IÍ]TULO\s+([IVXLCDM]+)$",
    re.IGNORECASE
)

ARTICLE_PATTERN = re.compile(
    r"^Artigo\s+(.+?º(?:-[A-Z])?)$",
    re.IGNORECASE
)


## 5. Test article identifier coverage

Check the article pattern against representative identifier formats found in the CRMP.


In [5]:
test_articles = [
    "Artigo A/1.º",
    "Artigo A-1/1.º",
    "Artigo A-2/5.º",
    "Artigo D-4/14.º-M",
    "Artigo H/3.º-A",
]

for value in test_articles:
    match = ARTICLE_PATTERN.match(value)

    print(
        value,
        "->",
        match.group(1) if match else None
    )


Artigo A/1.º -> A/1.º
Artigo A-1/1.º -> A-1/1.º
Artigo A-2/5.º -> A-2/5.º
Artigo D-4/14.º-M -> D-4/14.º-M
Artigo H/3.º-A -> H/3.º-A


## 6. Inspect detected headings

Scan the corpus and print lines recognized as structural legal headings.


In [6]:
# Scan pages in source order to validate structural heading detection.
for page in pages:
    for line in page["text"].splitlines():
        line = line.strip()

        if PART_PATTERN.match(line):
            print(page["page"], "PART :", line)

        elif TITLE_PATTERN.match(line):
            print(page["page"], "TITLE:", line)

        elif CHAPTER_PATTERN.match(line):
            print(page["page"], "CHAPTER:", line)

        elif ARTICLE_PATTERN.match(line):
            print(page["page"], "ARTICLE:", line)


22 PART : Parte A
22 PART : PARTE A
22 ARTICLE: Artigo A/1.º
22 ARTICLE: Artigo A/2.º
22 TITLE: TÍTULO I
22 ARTICLE: Artigo A-1/1.º
23 PART : Parte A
23 ARTICLE: Artigo A-1/2.º
23 ARTICLE: Artigo A-1/3.º
23 ARTICLE: Artigo A-1/4.º
23 ARTICLE: Artigo A-1/5.º
24 PART : Parte A
24 ARTICLE: Artigo A-1/6.º
24 ARTICLE: Artigo A-1/7.º
25 PART : Parte A
26 PART : Parte A
26 TITLE: TÍTULO II
26 ARTICLE: Artigo A-2/1.º
27 PART : Parte A
27 ARTICLE: Artigo A-2/2.º
27 ARTICLE: Artigo A-2/3.º
27 ARTICLE: Artigo A-2/4.º
28 PART : Parte A
28 ARTICLE: Artigo A-2/5.º
29 PART : Parte A
29 ARTICLE: Artigo A-2/6.º
29 ARTICLE: Artigo A-2/7.º
29 ARTICLE: Artigo A-2/8.º
30 PART : Parte A
30 ARTICLE: Artigo A-2/9.º
30 ARTICLE: Artigo A-2/10.º
30 ARTICLE: Artigo A-2/11.º
31 PART : Parte A
31 ARTICLE: Artigo A-2/12.º
32 PART : Parte A
32 ARTICLE: Artigo A-2/13.º
32 ARTICLE: Artigo A-2/13.º-A
33 PART : Parte A
33 ARTICLE: Artigo A-2/14.º
33 ARTICLE: Artigo A-2/15.º
33 ARTICLE: Artigo A-2/15.º-A
34 PART : Parte A

## 7. Create a heading classifier

Return a normalized heading type and value for any recognized structural line.


In [7]:
def identify_heading(line):
    line = line.strip()

    match = PART_PATTERN.match(line)

    if match:
        return "part", match.group(1)

    match = TITLE_PATTERN.match(line)

    if match:
        return "title", match.group(1)

    match = CHAPTER_PATTERN.match(line)

    if match:
        return "chapter", match.group(1)

    match = ARTICLE_PATTERN.match(line)

    if match:
        return "article", match.group(1)

    return None, None


## 8. Validate heading classification

Test the classifier against representative parts, titles, chapters, and articles.


In [8]:
examples = [
    "PARTE A",
    "TÍTULO I",
    "CAPÍTULO II",
    "Artigo A-1/3.º",
    "Texto normal"
]

for line in examples:
    print(line, "->", identify_heading(line))


PARTE A -> ('part', 'A')
TÍTULO I -> ('title', 'I')
CAPÍTULO II -> ('chapter', 'II')
Artigo A-1/3.º -> ('article', 'A-1/3.º')
Texto normal -> (None, None)


## 9. Define the article parser

Implement the stateful parser that carries hierarchy context across pages and builds article records.


In [9]:
def parse_articles(pages):

    articles = []

    # These variables carry the active legal hierarchy across pages.
    current_part = None
    current_title = None
    current_chapter = None

    current_article = None

    expecting_article_title = False

    for page in pages:

        page_number = page["page"]

        lines = [
            line.strip()
            for line in page["text"].splitlines()
            if line.strip()
        ]

        i = 0

        while i < len(lines):

            line = lines[i]

            heading_type, heading_value = identify_heading(line)

           # PART
            # ---------------------------------

            if heading_type == "part":
                current_part = heading_value
                i += 1
                continue

           # TITLE
            # ---------------------------------

            if heading_type == "title":
                current_title = heading_value
                current_chapter = None
                i += 1
                continue

           # CHAPTER
            # ---------------------------------

            if heading_type == "chapter":
                current_chapter = heading_value
                i += 1
                continue

           # ARTICLE
            # ---------------------------------

            if heading_type == "article":

                # Save the previous article before starting the next one.
                if current_article is not None:

                    current_article["text"] = (
                        "\n".join(current_article["text_lines"])
                        .strip()
                    )

                    del current_article["text_lines"]

                    articles.append(current_article)

                current_article = {
                    "part": current_part,
                    "title": current_title,
                    "chapter": current_chapter,
                    "article": heading_value,
                    "article_title": None,
                    "page_start": page_number,
                    "page_end": page_number,
                    "text_lines": []
                }

                expecting_article_title = True

                i += 1
                continue

           # Article title
            # ---------------------------------

            if (
                current_article is not None
                and expecting_article_title
            ):

                current_article["article_title"] = line

                expecting_article_title = False

                i += 1
                continue

           # Article body
            # ---------------------------------

            if current_article is not None:

                current_article["text_lines"].append(line)

                current_article["page_end"] = page_number

            i += 1

    # guardar último artigo
    if current_article is not None:

        current_article["text"] = (
            "\n".join(current_article["text_lines"])
            .strip()
        )

        del current_article["text_lines"]

        articles.append(current_article)

    return articles


## 10. Parse all articles

Run the parser over the cleaned document and report the number of identified articles.


In [10]:
articles = parse_articles(pages)

print(f"Artigos encontrados: {len(articles)}")


Artigos encontrados: 1440


## 11. Preview structured articles

Inspect the hierarchy, identifiers, page ranges, and text of the first parsed records.


In [11]:
for article in articles[:10]:

    print("=" * 100)

    print("Parte:", article["part"])
    print("Título:", article["title"])
    print("Capítulo:", article["chapter"])

    print(
        "Artigo:",
        article["article"]
    )

    print(
        "Epígrafe:",
        article["article_title"]
    )

    print(
        "Páginas:",
        article["page_start"],
        "-",
        article["page_end"]
    )

    print()

    print(article["text"][:1000])

    print()


Parte: A
Título: None
Capítulo: None
Artigo: A/1.º
Epígrafe: Objeto do código
Páginas: 22 - 22

1 – O presente código consagra as disposições regulamentares com eficácia externa em
vigor na área do Município do Porto nos seguintes domínios:
a) Urbanismo;
b) Ambiente;
c) Gestão do espaço público;
d) Intervenção municipal sobre o exercício de atividades privadas;
e) Gestão de recursos;
f) Taxas e outras receitas municipais;
g) Fiscalização e sancionamento de infrações.
2 – Esta codificação não prejudica a existência, nos domínios referidos, de disposições
regulamentares complementares ao presente código, nele devidamente referenciadas.

Parte: A
Título: None
Capítulo: None
Artigo: A/2.º
Epígrafe: Objeto da Parte A
Páginas: 22 - 22

A Parte A consagra:
a) No Título I, os princípios gerais inspiradores do código, que, para além dos
princípios gerais de fonte constitucional e legal, devem orientar o Município no
desenvolvimento da sua atividade;
b) No Título II, as disposições comuns aplicá

## 12. Create an article lookup helper

Define a convenient way to retrieve a parsed article by its legal identifier.


In [12]:
def find_article(articles, article_number):

    return next(
        (
            article
            for article in articles
            if article["article"] == article_number
        ),
        None
    )


## 13. Inspect a known article

Retrieve a representative article and review its complete structured record.


In [14]:
article = find_article(
    articles,
    "A-1/1.º"
)

print(article["article"])
print(article["article_title"])
print()
print(article["text"])


A-1/1.º
Prossecução do interesse público

1 – Toda a atividade municipal dirige-se à prossecução do interesse público, visando
assegurar a adequada harmonização dos interesses particulares com o interesse geral.
Parte Geral
2 – Incumbe ao Município fazer prevalecer as exigências impostas pelo interesse público
sobre os interesses particulares, nas condições previstas na lei, no presente código e
demais regulamentação aplicável.


## 14. Check for missing article titles

Identify articles for which no subtitle or epigraph was captured.


In [15]:
missing_titles = [
    article
    for article in articles
    if not article["article_title"]
]

print(
    f"Artigos sem epígrafe: "
    f"{len(missing_titles)}"
)


Artigos sem epígrafe: 0


## 15. Check for empty article bodies

Detect article records without substantive text.


In [16]:
empty_articles = [
    article
    for article in articles
    if not article["text"].strip()
]

print(
    f"Artigos sem texto: "
    f"{len(empty_articles)}"
)


Artigos sem texto: 22


## 16. Count articles by part

Aggregate the parsed records across the top-level CRMP divisions.


In [17]:
from collections import Counter

parts = Counter(
    article["part"]
    for article in articles
)

parts


Counter({'E': 647,
         'D': 474,
         'C': 107,
         'B': 68,
         'H': 52,
         'G': 38,
         'A': 28,
         'F': 26})

## 17. Display the part distribution

Print the number of parsed articles assigned to each part.


In [18]:
for part, count in sorted(parts.items()):
    print(
        f"Parte {part}: "
        f"{count} artigos"
    )


Parte A: 28 artigos
Parte B: 68 artigos
Parte C: 107 artigos
Parte D: 474 artigos
Parte E: 647 artigos
Parte F: 26 artigos
Parte G: 38 artigos
Parte H: 52 artigos


## 18. Define stable article IDs

Normalize legal article identifiers into machine-friendly IDs for downstream references.


In [19]:
def create_article_id(article):

    value = article["article"]

    # Normalize legal notation into a stable identifier suitable for joins and vector IDs.
    value = value.lower()

    value = value.replace("º", "")
    value = value.replace(".", "")
    value = value.replace("/", "_")
    value = value.replace("-", "_")

    return f"crmp_{value}"


## 19. Assign IDs to every article

Add the generated stable identifier to each parsed record.


In [20]:
for article in articles:
    article["id"] = create_article_id(article)


## 20. Verify an assigned ID

Inspect a representative article after ID generation.


In [21]:
article = find_article(
    articles,
    "A-1/1.º"
)

print(article["id"])


crmp_a_1_1


## 21. Build the article dataset

Combine source metadata with the complete collection of structured articles.


In [22]:
output = {
    "document": data["document"],
    "source_file": data["source_file"],
    "num_articles": len(articles),
    "articles": articles
}


## 22. Save the parsed articles

Write the article dataset as UTF-8 JSON for the chunking stage.


In [23]:
OUTPUT_FILE.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        ensure_ascii=False,  # Preserve Portuguese legal text in readable form.
        indent=2
    )

print(
    f"Ficheiro criado: "
    f"{OUTPUT_FILE}"
)


Ficheiro criado: c:\Users\user\Documents\GitHub\legal-rag-pt\data\processed\crmp_articles.json
